# CE49X: Introduction to Computational Thinking and Data Science for Civil Engineers
## Week 7: Naive Bayes Classification

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

Based on *Python Data Science Handbook* by Jake VanderPlas  
Chapter 5: Machine Learning (Section 5.05 - Naive Bayes Classification)  
[https://jakevdp.github.io/PythonDataScienceHandbook/](https://jakevdp.github.io/PythonDataScienceHandbook/)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Table of Contents

1. [Introduction to Bayesian Classification](#1.-Introduction-to-Bayesian-Classification)
   - What is Naive Bayes? (with intuitive example)
   - Bayes's theorem (with worked numerical example)
   - The "Naive" assumption explained
   - Complete formula + hand computation exercise
2. [Gaussian Naive Bayes](#2.-Gaussian-Naive-Bayes)
   - Structural health classification example
   - Decision boundaries and probability predictions
3. [Multinomial Naive Bayes](#3.-Multinomial-Naive-Bayes)
   - Bag of Words and TF-IDF
   - 20 Newsgroups text classification
4. [When to Use Naive Bayes](#4.-When-to-Use-Naive-Bayes)
5. [Linear Regression](#5.-Linear-Regression)
6. [Summary and Next Steps](#6.-Summary-and-Next-Steps)

---
## 1. Introduction to Bayesian Classification

### What is Naive Bayes?

Imagine you are a structural engineer inspecting a bridge. You notice some cracks, measure the deflection, and check the vibration frequency. Based on your **experience** (past data), you mentally estimate: *"Given these measurements, this bridge is probably in fair condition."*

That reasoning -- combining prior knowledge with observed evidence to make a judgment -- is exactly what **Naive Bayes** does, but mathematically.

**What is it?**
- A family of simple **probabilistic classifiers** -- algorithms that predict which category (class) something belongs to
- Based on **Bayes' theorem**, a fundamental rule of probability theory
- Called "naive" because it makes a simplifying assumption (we'll explain this soon)
- Despite being simple, it's surprisingly effective in many real-world applications

**Where is it used?**
- **Spam filtering**: Is this email spam or not? (one of the earliest ML applications in production)
- **Medical diagnosis**: Given symptoms, what disease is most likely?
- **Document categorization**: What topic does this article belong to?
- **Sentiment analysis**: Is this product review positive or negative?
- **Structural health monitoring**: Is this structure in good, fair, or poor condition?

> **Key Insight: Why Start Here?**  
> Naive Bayes is an ideal first classification algorithm because it is fast, intuitive, requires very little data, and introduces the core idea of **probabilistic reasoning** that underlies much of machine learning.

### Think Like Naive Bayes: An Everyday Example

Before we dive into math, let's build intuition with a simple scenario.

**Scenario:** You receive an email. You want to decide: **spam** or **not spam**?

Here's how you (and Naive Bayes) reason:

1. **Prior knowledge (before reading the email):**  
   From experience, about 40% of your emails are spam. So before even looking, there's a 40% chance it's spam.

2. **Look at the evidence (the words in the email):**  
   The email contains the words "free", "winner", and "click".  
   - The word "free" appears in 80% of spam emails but only 5% of legitimate emails.  
   - The word "winner" appears in 70% of spam but only 2% of legitimate emails.  
   - The word "click" appears in 60% of spam but only 10% of legitimate emails.

3. **Combine the evidence:**  
   Spam score: 0.40 x 0.80 x 0.70 x 0.60 = 0.1344  
   Not-spam score: 0.60 x 0.05 x 0.02 x 0.10 = 0.00006  

4. **Decision:** The spam score is **~2200x higher** than the not-spam score. This email is almost certainly spam!

> **Key Insight: The Core Idea**  
> Naive Bayes multiplies the **prior probability** of each class by the **likelihood** of each observed feature given that class, then picks the class with the highest score. That's it!

### Bayes's Theorem

Now let's formalize the intuition from the spam example. Bayes's theorem is a mathematical rule for updating our beliefs when we see new evidence.

**The Formula:**

$$
P(L \mid \text{features}) = \frac{P(\text{features} \mid L) \cdot P(L)}{P(\text{features})}
$$

**Let's break down each part in plain language:**

| Symbol | Name | Plain Language | Spam Example |
|--------|------|---------------|---------------|
| $P(L \mid \text{features})$ | **Posterior** | What we want to know: probability of label $L$ *after* seeing the evidence | P(spam \| "free", "winner") |
| $P(\text{features} \mid L)$ | **Likelihood** | How probable is this evidence if the label really is $L$? | P("free", "winner" \| spam) |
| $P(L)$ | **Prior** | How probable is label $L$ *before* seeing any evidence? | P(spam) = 0.40 |
| $P(\text{features})$ | **Evidence** | How probable is this evidence overall? (a normalizing constant) | P("free", "winner") |

> **Definition: Posterior, Likelihood, and Prior**  
> The posterior is proportional to the likelihood times the prior:  
> $$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$  
> The evidence $P(\text{features})$ is the same for all classes, so we can ignore it when comparing classes.

### [QUICK] Worked Example: Bayes's Theorem by Hand

Let's make Bayes's theorem concrete with a civil engineering example.

**Scenario:** A structural inspector measures a crack length of **8 mm** in a beam. Based on historical data:
- 70% of beams are in **Good** condition, 30% are in **Poor** condition (priors)
- In Good beams, crack lengths follow a distribution where P(8 mm | Good) = 0.02
- In Poor beams, crack lengths follow a distribution where P(8 mm | Poor) = 0.15

**Question:** What is the probability that this beam is in Poor condition?

In [ ]:
# Worked Example: Bayes's Theorem step by step

# Step 1: Define the priors (what we know before seeing the crack)
P_good = 0.70   # 70% of beams are in Good condition
P_poor = 0.30   # 30% of beams are in Poor condition
print(f"Step 1 - Priors: P(Good) = {P_good}, P(Poor) = {P_poor}")

# Step 2: Define the likelihoods (how likely is an 8mm crack for each condition?)
P_crack_given_good = 0.02  # An 8mm crack is rare in Good beams
P_crack_given_poor = 0.15  # An 8mm crack is common in Poor beams
print(f"Step 2 - Likelihoods: P(8mm | Good) = {P_crack_given_good}, P(8mm | Poor) = {P_crack_given_poor}")

# Step 3: Compute the numerator for each class (Likelihood × Prior)
score_good = P_crack_given_good * P_good
score_poor = P_crack_given_poor * P_poor
print(f"\nStep 3 - Unnormalized scores:")
print(f"  Good: {P_crack_given_good} × {P_good} = {score_good:.4f}")
print(f"  Poor: {P_crack_given_poor} × {P_poor} = {score_poor:.4f}")

# Step 4: Normalize to get true probabilities (divide by total evidence)
evidence = score_good + score_poor
P_good_given_crack = score_good / evidence
P_poor_given_crack = score_poor / evidence
print(f"\nStep 4 - Posterior probabilities (after normalization):")
print(f"  P(Good | 8mm crack) = {score_good:.4f} / {evidence:.4f} = {P_good_given_crack:.4f} ({P_good_given_crack*100:.1f}%)")
print(f"  P(Poor | 8mm crack) = {score_poor:.4f} / {evidence:.4f} = {P_poor_given_crack:.4f} ({P_poor_given_crack*100:.1f}%)")

# Step 5: Decision
print(f"\nDecision: Classify as {'Poor' if P_poor_given_crack > P_good_given_crack else 'Good'}")
print(f"\nNotice: Even though 70% of beams are Good (strong prior),")
print(f"the 8mm crack is so much more likely in Poor beams that the posterior flips!")
print(f"The prior was 30% Poor, but the posterior is {P_poor_given_crack*100:.1f}% Poor.")

### Bayes's Theorem for Classification

In the worked example above, we compared two scores and picked the higher one. Let's formalize this.

**Binary Classification (two classes):**

For two classes $L_1$ and $L_2$, we compute the posterior ratio:

$$
\frac{P(L_1 \mid \text{features})}{P(L_2 \mid \text{features})} = \frac{P(\text{features} \mid L_1) \, P(L_1)}{P(\text{features} \mid L_2) \, P(L_2)}
$$

**Why this simplification works:**
- The denominator $P(\text{features})$ is the same for both classes, so it cancels out in the ratio
- We only need to compute the **numerator** (likelihood $\times$ prior) for each class
- Choose the class with the highest product -- exactly what we did in the crack length example!
- This naturally extends to **multi-class** problems: just compute the score for each class and pick the winner

> **Key Insight: Decision Rule**  
> **Predict class** $L_{\text{max}} = \arg\max_L \; P(\text{features} \mid L) \, P(L)$  
>  
> In words: pick the class that gives the highest "likelihood $\times$ prior" score.

### Generative vs. Discriminative Models

Before we go further, it's helpful to understand where Naive Bayes fits in the bigger picture of classification algorithms.

There are two fundamentally different philosophies for building classifiers:

| | **Generative Models** | **Discriminative Models** |
|---|---|---|
| **Philosophy** | Learn *how each class generates data* | Learn *what separates the classes* |
| **What they model** | $P(\text{features} \mid L)$ for each class | $P(L \mid \text{features})$ directly |
| **Analogy** | Learning to paint like Monet vs. Picasso | Learning to tell Monet apart from Picasso |
| **Can generate new samples?** | Yes (since they model the data distribution) | No |
| **Examples** | **Naive Bayes**, Gaussian Mixture Models | Logistic Regression, SVM, Neural Networks |
| **CE Example** | Learn what "Good" and "Poor" beams look like in terms of crack, deflection, vibration distributions | Learn the boundary line/surface separating Good from Poor |

> **Example: Why Does This Matter?**  
> Generative models need stronger assumptions (e.g., "features follow a Gaussian distribution") but can work with less data. Discriminative models make fewer assumptions but typically need more data. Naive Bayes is a generative model.

### The "Naive" Assumption

Here's where the "naive" part comes in -- and it's the key idea that makes Naive Bayes fast and simple.

**The assumption:** Features are **conditionally independent** given the class label.

In math:

$$
P(f_1, f_2, \ldots, f_n \mid L) = P(f_1 \mid L) \cdot P(f_2 \mid L) \cdot \ldots \cdot P(f_n \mid L)
$$

**What does this mean in plain language?**

It means: *"Once you know the class, knowing one feature tells you nothing about the other features."*

**Concrete example:** If a beam is in **Poor** condition, knowing that its crack length is 12 mm doesn't change our estimate of its deflection. We treat each measurement as providing **independent** evidence about the condition.

**Why is this "naive"?**
- In reality, features are often correlated (a beam with long cracks probably also has large deflections)
- The assumption is almost never true in practice!

**So why does it work?**
- For **classification**, we only need to get the *ranking* of classes right, not the exact probabilities
- Even if the probability estimates are wrong, the *most probable class* is often still correct
- The assumption makes computation **dramatically** simpler (see below)

**The computational payoff:**

| | Without independence | With independence |
|---|---|---|
| **Parameters to estimate** | One joint probability for every combination of feature values | One probability per feature per class |
| **With 10 features, 2 classes** | $2 \times 2^{10} = 2048$ parameters | $2 \times 10 = 20$ parameters |
| **With 1000 features (text)** | Impossible | $2 \times 1000 = 2000$ parameters |

> **Key Insight: The Trade-off**  
> We trade a potentially wrong assumption for a model that is fast, needs very little data, and still works surprisingly well in practice -- especially for text classification and high-dimensional data.

### Complete Naive Bayes Formula

Let's put everything together into the full Naive Bayes classification algorithm.

**Given:** A new data point with features $\mathbf{x} = (f_1, f_2, \ldots, f_n)$

**Goal:** Find the class $L$ with the highest posterior probability:

$$
P(L \mid f_1, \ldots, f_n) \propto P(L) \prod_{i=1}^{n} P(f_i \mid L)
$$

**Step-by-step algorithm:**

**Training phase** (learning from labeled data):
1. **Compute priors:** For each class $L$, count how often it appears: $P(L) = \frac{\text{count}(L)}{\text{total samples}}$
2. **Compute likelihoods:** For each class $L$ and each feature $f_i$, learn $P(f_i \mid L)$ from the training data

**Prediction phase** (classifying a new sample):
3. For **each candidate class** $L$, compute: $\text{score}(L) = P(L) \times P(f_1 \mid L) \times P(f_2 \mid L) \times \cdots \times P(f_n \mid L)$
4. **Predict** the class with the highest score

> **Example: Efficiency**  
> With $n$ features and $k$ classes, we only need to estimate:
> - $k$ prior probabilities
> - $k \times n$ feature likelihoods  
>
> Compare this to $k \times 2^n$ parameters without the independence assumption!

**But how do we compute** $P(f_i \mid L)$?

That depends on what kind of data we have. This leads to **different variants** of Naive Bayes:
- **Gaussian NB**: Features are continuous numbers → model each as a Gaussian (bell curve)
- **Multinomial NB**: Features are word counts/frequencies → model each as a multinomial distribution
- **Bernoulli NB**: Features are binary (yes/no) → model each as a Bernoulli distribution

We'll study the first two in detail.

### [TOGETHER] Naive Bayes Classification by Hand

Before using scikit-learn, let's implement Naive Bayes from scratch on a tiny dataset to see exactly what happens at each step. This will solidify your understanding before we scale up.

In [ ]:
# Let's classify structural conditions by hand!
# Tiny dataset: 6 beams with crack length and deflection measurements

import pandas as pd
import numpy as np

# Training data
data = pd.DataFrame({
    'Crack (mm)':   [1.0, 2.0, 1.5,  8.0, 10.0, 9.0],
    'Deflect (mm)': [3.0, 4.0, 3.5, 15.0, 18.0, 16.0],
    'Condition':    ['Good', 'Good', 'Good', 'Poor', 'Poor', 'Poor']
})
print("Training data:")
print(data)
print()

# ──────────────────────────────────────────────────
# STEP 1: Compute priors
# ──────────────────────────────────────────────────
n_good = (data['Condition'] == 'Good').sum()
n_poor = (data['Condition'] == 'Poor').sum()
n_total = len(data)
P_good = n_good / n_total
P_poor = n_poor / n_total
print(f"STEP 1 — Priors:")
print(f"  P(Good) = {n_good}/{n_total} = {P_good:.3f}")
print(f"  P(Poor) = {n_poor}/{n_total} = {P_poor:.3f}")
print()

# ──────────────────────────────────────────────────
# STEP 2: Compute Gaussian parameters (mean, std) for each feature per class
# ──────────────────────────────────────────────────
good_data = data[data['Condition'] == 'Good']
poor_data = data[data['Condition'] == 'Poor']

stats = {}
for condition, subset in [('Good', good_data), ('Poor', poor_data)]:
    stats[condition] = {}
    for feat in ['Crack (mm)', 'Deflect (mm)']:
        mu = subset[feat].mean()
        sigma = subset[feat].std(ddof=0)  # population std, matching sklearn
        stats[condition][feat] = (mu, sigma)

print("STEP 2 — Gaussian parameters (mean, std) per class:")
for condition in ['Good', 'Poor']:
    print(f"  {condition}:")
    for feat in ['Crack (mm)', 'Deflect (mm)']:
        mu, sigma = stats[condition][feat]
        print(f"    {feat}: mean = {mu:.2f}, std = {sigma:.2f}")
print()

# ──────────────────────────────────────────────────
# STEP 3: Classify a NEW beam with crack = 7mm, deflection = 12mm
# ──────────────────────────────────────────────────
new_crack = 7.0
new_deflect = 12.0
print(f"STEP 3 — Classify new beam: crack = {new_crack} mm, deflection = {new_deflect} mm")
print()

from scipy.stats import norm

for condition in ['Good', 'Poor']:
    prior = P_good if condition == 'Good' else P_poor
    mu_c, sig_c = stats[condition]['Crack (mm)']
    mu_d, sig_d = stats[condition]['Deflect (mm)']
    
    # Likelihood for each feature
    lik_crack = norm.pdf(new_crack, mu_c, sig_c)
    lik_deflect = norm.pdf(new_deflect, mu_d, sig_d)
    
    score = prior * lik_crack * lik_deflect
    print(f"  {condition}:")
    print(f"    Prior:              P({condition}) = {prior:.3f}")
    print(f"    P(crack=7 | {condition}):  {lik_crack:.6f}  (Gaussian with mean={mu_c:.1f}, std={sig_c:.2f})")
    print(f"    P(defl=12 | {condition}):  {lik_deflect:.6f}  (Gaussian with mean={mu_d:.1f}, std={sig_d:.2f})")
    print(f"    Score = {prior:.3f} × {lik_crack:.6f} × {lik_deflect:.6f} = {score:.10f}")
    print()

# Compute normalized probabilities
score_good = P_good * norm.pdf(new_crack, *stats['Good']['Crack (mm)']) * norm.pdf(new_deflect, *stats['Good']['Deflect (mm)'])
score_poor = P_poor * norm.pdf(new_crack, *stats['Poor']['Crack (mm)']) * norm.pdf(new_deflect, *stats['Poor']['Deflect (mm)'])
total = score_good + score_poor

print(f"STEP 4 — Normalize:")
print(f"  P(Good | data) = {score_good/total:.4f} ({score_good/total*100:.1f}%)")
print(f"  P(Poor | data) = {score_poor/total:.4f} ({score_poor/total*100:.1f}%)")
print(f"\n  → Prediction: {'Poor' if score_poor > score_good else 'Good'}")
print(f"\nThis is exactly what sklearn's GaussianNB does internally!")

---
## 2. Gaussian Naive Bayes

### Gaussian Naive Bayes: The Idea

Now that we've seen Naive Bayes by hand, let's study the most common variant for numerical data: **Gaussian Naive Bayes**.

**Core assumption:** Each feature within each class follows a **Gaussian (normal) distribution** -- the familiar bell curve.

$$
P(f_i \mid L) = \frac{1}{\sqrt{2\pi\sigma_{L}^2}} \exp\left(-\frac{(f_i - \mu_L)^2}{2\sigma_L^2}\right)
$$

**What this means intuitively:**
- For each class (e.g., "Good", "Fair", "Poor"), we fit a bell curve to each feature
- When a new data point arrives, we check: *"Under which class's bell curve is this value most probable?"*
- We do this for every feature, multiply the results together, and pick the winning class

**When to use Gaussian NB:**
- Features are continuous numerical values (measurements, sensor readings)
- Features approximately follow normal distributions (common with physical measurements)
- You need a fast baseline for a classification problem

**Parameters to estimate per class $L$ and feature $i$:**
- $\mu_{L,i}$ = mean of feature $i$ for class $L$
- $\sigma_{L,i}$ = standard deviation of feature $i$ for class $L$

> **Key Insight: Simplicity**  
> Training Gaussian NB is literally just computing the mean and standard deviation of each feature for each class. That's why it's so fast!

### Gaussian Naive Bayes: Training

**Training Process:**

Given training data with features $\mathbf{X}$ and labels $\mathbf{y}$:

**Step 1: Compute class priors**

$$
P(L) = \frac{\text{number of samples with label } L}{\text{total number of samples}}
$$

**Step 2: For each class $L$ and feature $i$, compute:**
- Mean: $\mu_{L,i} = \text{mean of feature } i \text{ for samples with label } L$
- Std dev: $\sigma_{L,i} = \text{std dev of feature } i \text{ for samples with label } L$

> **Key Insight: Key Point**  
> Training is just computing sample statistics -- very fast!

### Gaussian Naive Bayes in Scikit-Learn

**Simple Implementation:**

```python
from sklearn.naive_bayes import GaussianNB

# Create the model
model = GaussianNB()

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Get probability estimates
y_prob = model.predict_proba(X_test)
```

**Key Methods:**
- `fit(X, y)`: Train the model on data
- `predict(X)`: Predict class labels for new data
- `predict_proba(X)`: Get probability estimates for each class
- `score(X, y)`: Get accuracy on test data

### [TOGETHER] Example: Structural Health Classification

Let's apply Gaussian Naive Bayes to a realistic civil engineering problem.

**Problem statement:** During routine bridge inspections, engineers measure three quantities:
- **Crack length** (mm): how long is the longest crack?
- **Deflection** (mm): how much does the structure sag under load?
- **Vibration frequency** (Hz): what is the natural frequency? (lower = more damage)

Based on these measurements, classify the structure as **Good**, **Fair**, or **Poor**.

**Our approach (step by step):**
1. Generate synthetic (simulated) inspection data -- since we don't have real data, we'll create realistic-looking data
2. Visualize the data to understand the patterns
3. Split into training and test sets
4. Fit a Gaussian NB model (i.e., compute mean and std for each feature per class)
5. Evaluate how well the model classifies unseen structures

> **Example: Why synthetic data?**  
> In a classroom setting, we use synthetic data so we know the "ground truth" and can control the difficulty. The same workflow applies to real inspection databases -- only Step 1 changes (you load real data instead of generating it).

In [ ]:
# Generate synthetic structural health data
def generate_structural_data(n_samples_per_class=150):
    """
    Generate synthetic structural inspection data.
    
    Returns:
    - X: Feature matrix (crack_length, deflection, vibration)
    - y: Target labels (0=Good, 1=Fair, 2=Poor)
    """
    
    # Good condition: small cracks, small deflection, high frequency
    good_crack = np.random.normal(loc=2.0, scale=0.8, size=n_samples_per_class)
    good_deflection = np.random.normal(loc=5.0, scale=1.5, size=n_samples_per_class)
    good_vibration = np.random.normal(loc=15.0, scale=2.0, size=n_samples_per_class)
    good_data = np.column_stack([good_crack, good_deflection, good_vibration])
    good_labels = np.zeros(n_samples_per_class)
    
    # Fair condition: medium cracks, medium deflection, medium frequency
    fair_crack = np.random.normal(loc=6.0, scale=1.2, size=n_samples_per_class)
    fair_deflection = np.random.normal(loc=12.0, scale=2.0, size=n_samples_per_class)
    fair_vibration = np.random.normal(loc=10.0, scale=1.5, size=n_samples_per_class)
    fair_data = np.column_stack([fair_crack, fair_deflection, fair_vibration])
    fair_labels = np.ones(n_samples_per_class)
    
    # Poor condition: large cracks, large deflection, low frequency
    poor_crack = np.random.normal(loc=12.0, scale=1.5, size=n_samples_per_class)
    poor_deflection = np.random.normal(loc=20.0, scale=2.5, size=n_samples_per_class)
    poor_vibration = np.random.normal(loc=5.0, scale=1.2, size=n_samples_per_class)
    poor_data = np.column_stack([poor_crack, poor_deflection, poor_vibration])
    poor_labels = np.full(n_samples_per_class, 2)
    
    # Combine all data
    X = np.vstack([good_data, fair_data, poor_data])
    y = np.concatenate([good_labels, fair_labels, poor_labels])
    
    # Ensure non-negative values (physical constraint)
    X = np.abs(X)
    
    return X, y

# Generate data
X, y = generate_structural_data(n_samples_per_class=150)

# Create a DataFrame for better visualization
df = pd.DataFrame(X, columns=['Crack Length (mm)', 'Deflection (mm)', 'Vibration (Hz)'])
df['Condition'] = y
df['Condition_Label'] = df['Condition'].map({0: 'Good', 1: 'Fair', 2: 'Poor'})

print("Dataset Shape:", X.shape)
print("\nFirst few rows:")
print(df.head(10))
print("\nClass distribution:")
print(df['Condition_Label'].value_counts())

In [ ]:
# Visualize the data in 2D (crack length vs deflection)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Crack Length vs Deflection
for condition, label in [(0, 'Good'), (1, 'Fair'), (2, 'Poor')]:
    mask = df['Condition'] == condition
    axes[0].scatter(df.loc[mask, 'Crack Length (mm)'], 
                   df.loc[mask, 'Deflection (mm)'],
                   label=label, alpha=0.6, s=50)
axes[0].set_xlabel('Crack Length (mm)', fontsize=12)
axes[0].set_ylabel('Deflection (mm)', fontsize=12)
axes[0].set_title('Crack Length vs Deflection', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Crack Length vs Vibration
for condition, label in [(0, 'Good'), (1, 'Fair'), (2, 'Poor')]:
    mask = df['Condition'] == condition
    axes[1].scatter(df.loc[mask, 'Crack Length (mm)'], 
                   df.loc[mask, 'Vibration (Hz)'],
                   label=label, alpha=0.6, s=50)
axes[1].set_xlabel('Crack Length (mm)', fontsize=12)
axes[1].set_ylabel('Vibration (Hz)', fontsize=12)
axes[1].set_title('Crack Length vs Vibration', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Deflection vs Vibration
for condition, label in [(0, 'Good'), (1, 'Fair'), (2, 'Poor')]:
    mask = df['Condition'] == condition
    axes[2].scatter(df.loc[mask, 'Deflection (mm)'], 
                   df.loc[mask, 'Vibration (Hz)'],
                   label=label, alpha=0.6, s=50)
axes[2].set_xlabel('Deflection (mm)', fontsize=12)
axes[2].set_ylabel('Vibration (Hz)', fontsize=12)
axes[2].set_title('Deflection vs Vibration', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Good condition: small cracks, small deflection, high vibration frequency")
print("- Fair condition: medium values for all features")
print("- Poor condition: large cracks, large deflection, low vibration frequency")
print("- Classes are reasonably well-separated, suitable for Gaussian NB")

**How to read these scatter plots:**

Each dot represents one inspected structure. The color shows its true condition (Good/Fair/Poor). Ask yourself:

- **Are the clusters well-separated?** If yes, Naive Bayes will likely perform well -- the distributions don't overlap much.
- **Is there overlap between classes?** Overlapping regions are where the model will make mistakes.
- **Do the clusters look roughly elliptical (Gaussian)?** If the shapes are round/elliptical, our Gaussian assumption is reasonable.

Notice how structures in **Good** condition cluster at low crack length and low deflection, while **Poor** structures have high values for both. This is the pattern our model needs to learn.

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {int(cls)}: {cnt} samples")

In [ ]:
# Create and train the Gaussian Naive Bayes model
gnb_model = GaussianNB()
gnb_model.fit(X_train, y_train)

print("Gaussian Naive Bayes Model trained successfully!")
print("\nModel Parameters:")
print(f"Classes: {gnb_model.classes_}")
print(f"\nClass priors (P(class)):")
for i, prior in enumerate(gnb_model.class_prior_):
    condition_name = ['Good', 'Fair', 'Poor'][i]
    print(f"  {condition_name}: {prior:.3f}")

print(f"\nFeature means for each class (theta):")
feature_names = ['Crack Length', 'Deflection', 'Vibration']
for i, condition in enumerate(['Good', 'Fair', 'Poor']):
    print(f"\n  {condition} condition:")
    for j, feature in enumerate(feature_names):
        print(f"    {feature}: mu={gnb_model.theta_[i, j]:.2f}, sigma^2={gnb_model.var_[i, j]:.2f}")

**Understanding what the model learned:**

Look at the output above. The model has stored very simple information:

- **Class priors** $P(L)$: Each class has equal probability (0.333) because we generated equal numbers of each condition. In real data, these would reflect how common each condition is.

- **Feature means** ($\mu$): For example, the mean crack length for "Good" beams is ~2.0 mm, but for "Poor" beams it's ~12.0 mm. These means define the *center* of each class's bell curve.

- **Feature variances** ($\sigma^2$): These define how *spread out* each bell curve is. A large variance means the feature varies a lot within that class.

That's the entire model! When classifying a new structure, the model asks: *"This crack length -- is it more probable under the Good bell curve, the Fair bell curve, or the Poor bell curve?"* and does the same for each feature.

### Gaussian Naive Bayes: Visualization

The plot below shows **everything** about how Gaussian NB makes decisions. Let's understand each element:

- **Colored dots**: Training data points (each dot = one inspected structure)
- **Dashed ellipses**: The Gaussian distribution each class learned (showing mean $\pm$ 2 standard deviations). Think of these as the model's "mental picture" of what each class looks like.
- **Shaded regions**: Where each class wins (has the highest posterior probability). A new point falling in the green region would be classified as "Good".
- **Dashed gray lines**: The **decision boundaries** -- where two classes are equally probable. Points right on the boundary could go either way.

Notice that the boundaries are **curved** (quadratic), not straight lines. This is because each class has its own variance.

In [ ]:
# Visualize Gaussian NB: Ellipses for each class and decision boundaries
from matplotlib.patches import Ellipse

fig, ax = plt.subplots(figsize=(10, 7))

# Use first two features: Crack Length and Deflection
X_2d = X[:, :2]
model_2d = GaussianNB()
model_2d.fit(X_2d, y)

# Create mesh for decision boundary
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot decision regions
ax.contourf(xx, yy, Z, alpha=0.2, levels=2, cmap='viridis')
ax.contour(xx, yy, Z, levels=2, colors='gray', linewidths=1, linestyles='dashed')

# Plot data points
colors = ['green', 'orange', 'red']
labels = ['Good', 'Fair', 'Poor']
for i, (color, label) in enumerate(zip(colors, labels)):
    mask = y == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
              c=color, label=label, edgecolors='black', s=40, alpha=0.6)

# Draw ellipses for each class (mean +/- 2 std)
for i, (color, label) in enumerate(zip(colors, labels)):
    mean = model_2d.theta_[i]
    var = model_2d.var_[i]
    ell = Ellipse(xy=mean, width=4*np.sqrt(var[0]), height=4*np.sqrt(var[1]),
                  edgecolor=color, facecolor='none', linewidth=2, linestyle='--')
    ax.add_patch(ell)

ax.set_xlabel('Crack Length (mm)', fontsize=12)
ax.set_ylabel('Deflection (mm)', fontsize=12)
ax.set_title('Gaussian Naive Bayes: Decision Boundaries and Class Distributions',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Decision Boundaries

**What is a decision boundary?**

A decision boundary is the line (or curve) in feature space where two classes are *equally probable*. On one side of the boundary, Class A wins; on the other side, Class B wins.

**For Gaussian Naive Bayes:**
- Decision boundaries are **quadratic** (curved) in general
- They become **linear** (straight) only when all classes share the same variance
- This is different from logistic regression, which always has linear boundaries
- The curvature allows Gaussian NB to capture some non-linear patterns

> **Example: Mathematical Detail (optional)**  
> The log-posterior is a quadratic function of features:
>
> $$\log P(L \mid \mathbf{x}) \propto -\sum_i \frac{(f_i - \mu_{L,i})^2}{2\sigma_{L,i}^2} + \log P(L)$$
>
> Setting the log-posteriors of two classes equal gives a quadratic equation in the features, hence curved boundaries.

### Probability Predictions

**Why probabilities matter more than labels:**

Most classifiers just say "Good" or "Poor". But Naive Bayes gives you a **probability for each class**. This is extremely valuable in engineering applications:

- **Confidence**: A prediction of "Good" with 95% confidence is very different from "Good" with 51% confidence
- **Risk management**: You might take action if P(Poor) > 0.30, even if the most likely class is "Fair"
- **Triage**: Rank structures by P(Poor) to prioritize inspections
- **Custom thresholds**: In safety-critical applications, you might classify as "Poor" if P(Poor) > 0.20 (conservative) rather than P(Poor) > 0.50 (default)

**Using `predict_proba()`:**

| Sample | **P(Good)** | **P(Fair)** | **P(Poor)** | Prediction |
|:------:|:-----------:|:-----------:|:-----------:|:----------:|
| 1      | **0.85**    | 0.13        | 0.02        | Good (high confidence) |
| 2      | 0.10        | **0.75**    | 0.15        | Fair (moderate confidence) |
| 3      | 0.02        | 0.28        | **0.70**    | Poor (moderate confidence) |
| 4      | 0.05        | **0.48**    | 0.47        | Fair (very low confidence!) |

Sample 4 is a borderline case -- the model is almost equally split between Fair and Poor. An engineer should inspect this structure more carefully.

In [ ]:
# Make predictions on the test set
y_pred = gnb_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Get probability predictions
y_prob = gnb_model.predict_proba(X_test)

print("\nExample predictions (first 10 test samples):")
print(f"{'True':<6} {'Pred':<6} {'P(Good)':<10} {'P(Fair)':<10} {'P(Poor)':<10}")
print("-" * 52)
for i in range(10):
    true_label = ['Good', 'Fair', 'Poor'][int(y_test[i])]
    pred_label = ['Good', 'Fair', 'Poor'][int(y_pred[i])]
    print(f"{true_label:<6} {pred_label:<6} {y_prob[i, 0]:<10.3f} {y_prob[i, 1]:<10.3f} {y_prob[i, 2]:<10.3f}")

In [ ]:
# Compute and visualize confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Good', 'Fair', 'Poor'],
            yticklabels=['Good', 'Fair', 'Poor'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Condition', fontsize=12, fontweight='bold')
plt.ylabel('True Condition', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix - Gaussian Naive Bayes\nStructural Health Classification', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Good', 'Fair', 'Poor']))

### [PRACTICE] Your Turn: Predict New Structures

Now it's your turn! Below, we classify four new structures that weren't in the training data. Look at the measurements and try to predict the condition **before** looking at the model's output:

| Structure | Crack (mm) | Deflection (mm) | Vibration (Hz) | Your guess? |
|:---------:|:----------:|:----------------:|:--------------:|:-----------:|
| A         | 1.5        | 4.0              | 16.0           | ?           |
| B         | 5.5        | 11.0             | 10.5           | ?           |
| C         | 13.0       | 22.0             | 4.5            | ?           |
| D         | 3.0        | 8.0              | 13.0           | ?           |

**Hint:** Think about the ranges we saw for each class:
- Good: crack ~2, deflection ~5, vibration ~15
- Fair: crack ~6, deflection ~12, vibration ~10
- Poor: crack ~12, deflection ~20, vibration ~5

Structure D is interesting -- it's between Good and Fair. Pay attention to the probabilities!

In [ ]:
# [PRACTICE] Predict condition for new structural inspections
new_structures = np.array([
    [1.5, 4.0, 16.0],   # Should be Good
    [5.5, 11.0, 10.5],  # Should be Fair
    [13.0, 22.0, 4.5],  # Should be Poor
    [3.0, 8.0, 13.0],   # Borderline Good/Fair
])

predictions = gnb_model.predict(new_structures)
probabilities = gnb_model.predict_proba(new_structures)

print("Predictions for new structures:")
print(f"\n{'Crack':<8} {'Deflect':<8} {'Vibr':<8} {'Predicted':<12} {'P(Good)':<10} {'P(Fair)':<10} {'P(Poor)':<10}")
print("-" * 76)
for i, (structure, pred, prob) in enumerate(zip(new_structures, predictions, probabilities)):
    pred_label = ['Good', 'Fair', 'Poor'][int(pred)]
    print(f"{structure[0]:<8.1f} {structure[1]:<8.1f} {structure[2]:<8.1f} "
          f"{pred_label:<12} {prob[0]:<10.3f} {prob[1]:<10.3f} {prob[2]:<10.3f}")

### Gaussian Naive Bayes: Strengths and Limitations

| **Strengths** | **Limitations** |
|---|---|
| Very fast training and prediction | Assumes features are independent |
| Works well with small datasets | Assumes Gaussian distributions |
| Provides probabilistic predictions | Can be outperformed by more complex models |
| No hyperparameters to tune | Sensitive to feature scaling |
| Handles multi-class naturally | Poor if assumptions violated |
| Interpretable (can examine means/stds) | |

> **Key Insight: Best Practices**  
> - Check if features are approximately Gaussian (histograms, Q-Q plots)
> - Consider transforming features (log, Box-Cox) if heavily skewed
> - Compare with other algorithms on your specific dataset

---
## 3. Multinomial Naive Bayes

### Multinomial Naive Bayes: The Idea

Gaussian NB works great for continuous measurements (crack lengths, deflections). But what if our features are **counts or frequencies** -- like the number of times each word appears in a document?

This is where **Multinomial Naive Bayes** comes in.

**Core assumption:** Features represent **counts** or **frequencies** of discrete events:

$$
P(f_i \mid L) = \frac{\text{count of feature } i \text{ in class } L}{\text{total count of all features in class } L}
$$

**When to use:**
- Features are discrete counts (e.g., word frequencies in documents)
- Text classification (the most common use case by far)
- Any count-based data (e.g., number of defects per category)

**Key difference from Gaussian NB:**

| | Gaussian NB | Multinomial NB |
|---|---|---|
| Feature type | Continuous (measurements) | Discrete (counts/frequencies) |
| Distribution | Gaussian (bell curve) | Multinomial (probability of each category) |
| Feature values | Any real number | Non-negative (0, 1, 2, ...) |
| Typical use | Sensor data, physical measurements | Text classification, count data |

### Text Classification with Multinomial NB

**The fundamental challenge:** Machine learning models need *numbers*, but text is *words*. How do we convert text into a numerical feature vector?

**Solution: The Bag of Words representation**

The idea is simple (and a bit crude): treat each document as a "bag" (unordered collection) of words, and count how many times each word appears.

**Step-by-step process:**
1. **Build a vocabulary** of all unique words across all documents
2. **For each document**, count how many times each vocabulary word appears
3. **Represent** each document as a vector of these counts
4. **Ignore word order** entirely (hence "bag" of words, not "sequence" of words)

**Example:**

| | the | bridge | building | is | safe |
|---|:---:|:---:|:---:|:---:|:---:|
| "the bridge is safe" | 1 | 1 | 0 | 1 | 1 |
| "the building is safe" | 1 | 0 | 1 | 1 | 1 |

**Limitation:** "The dog bit the man" and "The man bit the dog" have identical representations! Despite this, bag of words works surprisingly well for topic classification.

> **Key Insight: Why It Works for Classification**  
> For classifying documents by topic, *which words appear* matters more than *in what order*. A document about space will contain words like "NASA", "orbit", "telescope" regardless of word order.

### TF-IDF: Improving Bag of Words

**Problem with raw word counts:**

Common words like "the", "is", "a" appear frequently in *every* document but carry almost no information about the topic. Using raw counts gives these uninformative words too much weight.

**Solution: TF-IDF (Term Frequency - Inverse Document Frequency)**

The idea: weight each word by both how often it appears in this document AND how rare it is across all documents.

$$
\text{TF-IDF}(t, d) = \underbrace{\text{TF}(t, d)}_{\text{How often in this doc?}} \times \underbrace{\text{IDF}(t)}_{\text{How rare across all docs?}}
$$

where:
- $\text{TF}(t, d)$ = frequency of term $t$ in document $d$
- $\text{IDF}(t) = \log\frac{\text{total documents}}{\text{documents containing } t}$

**Intuition through examples:**

| Word | TF (appears often?) | IDF (rare across docs?) | TF-IDF |
|------|:---:|:---:|:---:|
| "the" | High | Very low (appears everywhere) | **Low** -- not useful |
| "bridge" | Medium | Medium | **Medium** |
| "prestressed" | Low | Very high (rare, technical) | **High** -- very informative! |

> **Key Insight: The Best of Both Worlds**  
> TF-IDF automatically down-weights common words and up-weights distinctive words, giving the classifier better features to work with. In scikit-learn, `TfidfVectorizer` handles this entire transformation.

### Multinomial NB in Scikit-Learn

**Implementation with TF-IDF:**

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# Create a pipeline: vectorization -> classification
model = make_pipeline(
    TfidfVectorizer(),
    MultinomialNB()
)

# Train the model
model.fit(train_data, train_labels)

# Make predictions
predictions = model.predict(test_data)
```

**Pipeline advantages:** Combines preprocessing and classification into one object.

### [TOGETHER] Example: 20 Newsgroups Dataset

Let's apply Multinomial NB to a real-world text classification task.

**The 20 Newsgroups dataset** is a classic benchmark in NLP/ML. It contains ~18,000 posts from 20 different internet newsgroups (online discussion forums from the 1990s). Each post belongs to one topic.

**We'll use 4 categories to keep things manageable:**
- `talk.religion.misc` -- General religious discussions
- `soc.religion.christian` -- Christianity-specific discussions
- `sci.space` -- Space science and exploration
- `comp.graphics` -- Computer graphics

**Why these 4?** They present an interesting challenge:
- `sci.space` and `comp.graphics` are very different topics -- easy to separate
- `talk.religion.misc` and `soc.religion.christian` share similar vocabulary -- harder to separate
- This lets us see where the model succeeds and where it struggles

**Our pipeline:** Raw text → TF-IDF features → Multinomial NB → Predicted category

In [ ]:
# Select 4 categories for classification
categories = ['talk.religion.misc',
              'soc.religion.christian',
              'sci.space',
              'comp.graphics']

# Load training data
train_data = fetch_20newsgroups(subset='train', 
                                categories=categories,
                                remove=('headers', 'footers', 'quotes'),
                                random_state=42)

# Load test data
test_data = fetch_20newsgroups(subset='test', 
                               categories=categories,
                               remove=('headers', 'footers', 'quotes'),
                               random_state=42)

print(f"Training samples: {len(train_data.data)}")
print(f"Testing samples: {len(test_data.data)}")
print(f"\nCategories: {train_data.target_names}")
print(f"\nTraining set class distribution:")
unique, counts = np.unique(train_data.target, return_counts=True)
for i, (cls, cnt) in enumerate(zip(unique, counts)):
    print(f"  {train_data.target_names[i]}: {cnt} samples")

In [ ]:
# Display sample documents from each category
print("Sample documents from each category:\n")
for i, category in enumerate(train_data.target_names):
    # Find first document of this category
    idx = np.where(train_data.target == i)[0][0]
    print(f"Category: {category}")
    print(f"Document preview (first 300 chars):")
    print(train_data.data[idx][:300])
    print("\n" + "="*80 + "\n")

In [ ]:
# Create a pipeline with TF-IDF vectorization and Multinomial NB
text_model = make_pipeline(
    TfidfVectorizer(max_features=5000, stop_words='english'),
    MultinomialNB(alpha=0.1)
)

# Train the model
print("Training the text classifier...")
text_model.fit(train_data.data, train_data.target)
print("Training complete!")

# Make predictions on test set
y_pred_text = text_model.predict(test_data.data)

# Calculate accuracy
accuracy_text = accuracy_score(test_data.target, y_pred_text)
print(f"\nTest Set Accuracy: {accuracy_text:.4f} ({accuracy_text*100:.2f}%)")

### Confusion Matrix

**What is a confusion matrix?**

A confusion matrix is a table that shows how many predictions the model got right and wrong, broken down by class. It's the single most informative evaluation tool for classifiers.

**How to read it:**
- **Rows** = the true (actual) class
- **Columns** = the predicted class
- **Diagonal cells** (top-left to bottom-right) = correct predictions
- **Off-diagonal cells** = errors (misclassifications)
- Each row sums to the total number of test samples for that true class

**What to look for:**
- A perfect model has all counts on the diagonal and zeros everywhere else
- Large off-diagonal values reveal which classes the model confuses with each other
- Pay special attention to whether `talk.religion.misc` and `soc.religion.christian` get confused -- they share similar vocabulary!

In [ ]:
# Compute and visualize confusion matrix
cm_text = confusion_matrix(test_data.target, y_pred_text)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_text, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['religion.misc', 'christian', 'space', 'graphics'],
            yticklabels=['religion.misc', 'christian', 'space', 'graphics'],
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Category', fontsize=12, fontweight='bold')
plt.ylabel('True Category', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix - Multinomial Naive Bayes\n20 Newsgroups Classification', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print classification report
print("\nDetailed Classification Report:")
print(classification_report(test_data.target, y_pred_text, 
                          target_names=train_data.target_names))

### [LIVE] Making Predictions on New Text

Now comes the fun part! Let's feed **our own sentences** to the trained model and see what it predicts.

This is where you really get a sense for what the model has learned. Try to predict what category the model will choose before looking at the output. Think about: *which words in this sentence are distinctive for a particular topic?*

After the predictions, we'll also look at the **most informative words** for each category -- the words that most strongly signal a particular topic.

In [ ]:
# Function to predict category for custom text
def predict_category(text, model, target_names):
    """
    Predict the category of a given text.
    """
    pred = model.predict([text])[0]
    prob = model.predict_proba([text])[0]
    
    print(f"Text: '{text[:100]}...'" if len(text) > 100 else f"Text: '{text}'")
    print(f"Predicted category: {target_names[pred]}")
    print(f"Probabilities:")
    for name, p in zip(target_names, prob):
        print(f"  {name}: {p:.4f}")
    print()

# Test with custom strings
custom_texts = [
    "NASA launched a new space telescope to explore distant galaxies and exoplanets",
    "The Bible teaches us about love, forgiveness, and the path to salvation",
    "OpenGL and DirectX are popular graphics APIs for rendering 3D scenes",
    "What is the meaning of life according to different philosophical traditions?",
    "The Mars rover discovered evidence of ancient water on the red planet",
    "Ray tracing algorithms produce photorealistic images by simulating light physics"
]

print("Predictions for custom text samples:\n")
print("=" * 80 + "\n")
for text in custom_texts:
    predict_category(text, text_model, train_data.target_names)
    print("=" * 80 + "\n")

In [ ]:
# Analyze most informative features
vectorizer = text_model.named_steps['tfidfvectorizer']
classifier = text_model.named_steps['multinomialnb']

feature_names_text = np.array(vectorizer.get_feature_names_out())

# For each category, show top 15 most informative words
print("Top 15 most informative words for each category:\n")
for i, category in enumerate(train_data.target_names):
    # Get log probabilities for this category
    log_prob = classifier.feature_log_prob_[i]
    
    # Get top 15 features
    top_indices = np.argsort(log_prob)[-15:][::-1]
    top_features = feature_names_text[top_indices]
    
    print(f"{category}:")
    print(f"  {', '.join(top_features)}")
    print()

### Multinomial NB: Strengths and Limitations

| **Strengths** | **Limitations** |
|---|---|
| Excellent for text classification | Assumes feature independence (violated in text) |
| Very fast training and prediction | Ignores word order and context |
| Works well with high-dimensional sparse data | Requires non-negative features |
| Requires relatively little training data | Can be beaten by deep learning for very large datasets |
| Handles multi-class naturally | Sensitive to feature representation |
| Provides probability estimates | |

> **Key Insight: Practical Tips**  
> - Use TF-IDF instead of raw counts for better performance
> - Remove very common and very rare words (stop words, rare terms)
> - Consider n-grams (word pairs, triples) to capture some context

### Laplace Smoothing

**The Zero Probability Problem:**

What happens if a word appears in a test document but was **never seen** in training for a particular class?

For example, suppose the word "asteroid" never appeared in any `comp.graphics` training document. Then:

$$
P(\text{"asteroid"} \mid \text{graphics}) = 0
$$

Because Naive Bayes **multiplies** all feature probabilities, this single zero **kills the entire score** for that class -- no matter how much other evidence supports it!

$$
P(\text{graphics} \mid \text{document}) \propto P(\text{graphics}) \times \ldots \times \underbrace{P(\text{"asteroid"} \mid \text{graphics})}_{= 0} \times \ldots = 0
$$

**Solution: Laplace (Additive) Smoothing**

Add a small count $\alpha$ (usually 1) to every feature count, so nothing is ever zero:

$$
P(f_i \mid L) = \frac{\text{count}(f_i, L) + \alpha}{\text{total count}(L) + \alpha \cdot n\_features}
$$

**Effect:**
- With $\alpha = 1$: every word is treated as if it appeared at least once in every class
- With $\alpha < 1$ (e.g., 0.1): less aggressive smoothing -- words that truly never appeared get very small (but non-zero) probabilities
- Controlled by the `alpha` parameter in scikit-learn's `MultinomialNB(alpha=...)`

> **Key Insight: A Practical Necessity**  
> Laplace smoothing is not optional -- without it, Multinomial NB would fail on virtually any real dataset where some words only appear in some categories. This is why scikit-learn uses $\alpha = 1$ by default.

### Model Comparison: Gaussian NB vs. Multinomial NB

In [ ]:
# Create a comparison summary
comparison_data = {
    'Model': ['Gaussian NB (Structural)', 'Multinomial NB (Text)'],
    'Dataset': ['Structural Health (450 samples)', '20 Newsgroups (~2400 samples)'],
    'Features': ['3 continuous features', '5000 TF-IDF features'],
    'Classes': ['3 (Good/Fair/Poor)', '4 (newsgroup categories)'],
    'Accuracy': [f'{accuracy:.4f}', f'{accuracy_text:.4f}']
}

comparison_df = pd.DataFrame(comparison_data)
print("Model Comparison Summary:\n")
print(comparison_df.to_string(index=False))

print("\n" + "="*80)
print("\nKey Observations:")
print("- Gaussian NB works well for continuous numerical features")
print("- Multinomial NB excels at text classification with count-based features")
print("- Both models are fast, interpretable, and provide probabilistic predictions")
print("- Choice of variant depends on the nature of your features")

---
## 4. When to Use Naive Bayes

### Advantages of Naive Bayes

**Why Choose Naive Bayes?**

**1. Speed**
- Training: Just compute statistics (means, counts)
- Prediction: Simple multiplication and comparison
- Scales well to very large datasets

**2. Simplicity**
- Few or no hyperparameters to tune
- Easy to implement and understand
- Straightforward interpretation

**3. Probabilistic Predictions**
- Provides probability estimates, not just labels
- Useful for ranking or setting decision thresholds
- Natural uncertainty quantification

**4. Works with Limited Data**
- Can train with relatively small datasets
- Strong assumptions reduce data requirements

### When Naive Bayes Works Well

**Ideal Scenarios:**

**1. High-Dimensional Data**
- Text classification (thousands of features)
- As dimensionality increases, classes become more separated
- Independence assumption matters less

**2. Well-Separated Classes**
- When class distributions don't overlap much
- Decision boundaries are not critical

**3. Need for Baseline**
- Quick baseline before trying complex models
- Fast prototyping and iteration

**4. Real-Time Predictions**
- When speed is critical
- Spam filtering, sentiment analysis

### Limitations and When NOT to Use

**Cases Where Naive Bayes Struggles:**

**1. Strong Feature Dependencies**
- Violates independence assumption
- Example: height and weight are correlated
- Better: use models that capture correlations

**2. Complex Decision Boundaries**
- Gaussian NB has quadratic boundaries
- Cannot learn arbitrary non-linear boundaries
- Better: SVM with RBF kernel, neural networks

**3. Continuous Features with Non-Gaussian Distribution**
- Heavy-tailed, multi-modal, or skewed distributions
- Better: transform features or use different model

**4. When You Need the Best Possible Accuracy**
- Ensemble methods (Random Forest, Gradient Boosting) usually better
- Deep learning for very large datasets

### Naive Bayes vs. Other Classifiers

| **Aspect** | **Naive Bayes** | **Logistic Regression** | **SVM** |
|---|---|---|---|
| Speed | Very fast | Fast | Moderate |
| Accuracy | Good | Good to very good | Very good |
| High-dim data | Excellent | Good | Good |
| Interpretability | High | High | Medium |
| Probability | Native | Native | Calibration needed |
| Assumptions | Strong | Moderate | Weak |
| Hyperparameters | Very few | Few | Several |

> **Key Insight: Recommendation**  
> Start with Naive Bayes as a baseline, then try more complex models if needed.

### The Curse of Dimensionality (Helps NB!)

**Surprising Advantage:**

In high dimensions, Naive Bayes often performs better than expected.

**Why?**
- As dimensions increase, data points become more separated
- Classes occupy distinct regions of feature space
- Simple decision boundaries work better
- Feature correlations matter less

**Example: Text Classification**
- Vocabulary size: 10,000+ words
- Very high-dimensional sparse vectors
- Documents from different topics use different words
- Even with naive independence, classification works well

> **Example: Key Insight**  
> Sometimes simple assumptions + high dimensions > complex models + low dimensions

### Practical Guidelines

**Decision Framework for Using Naive Bayes:**

**Use Naive Bayes when:**
- You need a quick baseline or prototype
- You have text or count data (Multinomial NB)
- You have continuous features that are roughly Gaussian (Gaussian NB)
- Speed is critical (real-time predictions)
- You have limited training data
- You need probability estimates
- Interpretability is important

**Consider alternatives when:**
- Features are highly correlated
- You need maximum accuracy (competitions, critical applications)
- Decision boundaries are very complex
- You have lots of data and computational resources

### Variants of Naive Bayes in Scikit-Learn

**Different Types for Different Data:**

**1. GaussianNB**
- Continuous features following Gaussian distribution
- Example: sensor measurements, physical properties

**2. MultinomialNB**
- Discrete counts or frequencies
- Example: word counts in text, event counts

**3. BernoulliNB**
- Binary features (0/1, True/False)
- Example: word presence/absence in document

**4. ComplementNB**
- Modified version of MultinomialNB
- Better for imbalanced datasets

---
## 5. Linear Regression

### From Classification to Regression

We've now covered Naive Bayes, a powerful and efficient **classification** algorithm. For the remainder of this lecture, we introduce **Linear Regression** -- the most fundamental **regression** algorithm.

**What's the difference?**

| | **Classification** (e.g., Naive Bayes) | **Regression** (e.g., Linear Regression) |
|---|---|---|
| **Question** | *Which category does this belong to?* | *What is the numerical value?* |
| **Output** | Discrete labels (Good / Fair / Poor) | Continuous numbers (12.7 mm, 345 kN) |
| **Example** | Is this beam safe or damaged? | How much will this beam deflect? |
| **Foundation** | Bayes' theorem | Least squares optimization |

> **Key Insight: Complementary Tools**  
> Both are simple, fast, interpretable baseline models for their respective tasks. A civil engineer needs both: Naive Bayes to classify structural condition, and Linear Regression to predict exact deflections, loads, or costs.

### What is Linear Regression?

**Overview (extending concepts from Week 06):**

Linear Regression is arguably the **oldest and most widely used** statistical technique. We introduced the basics in Week 06; here we extend those concepts to more powerful variants.

**Key Characteristics:**
- One of the oldest statistical methods (Legendre, 1805; Gauss, 1809)
- Extremely fast to train -- has a closed-form solution
- Highly interpretable: each coefficient tells you the effect of one feature
- Serves as the **baseline** against which more complex regression models are compared

**Civil Engineering Applications:**
- Predicting structural responses (deflection, stress) from loads
- Estimating material properties (strength vs. water-cement ratio)
- Modeling traffic flow (volume vs. time of day, lane count)
- Cost estimation (project cost vs. area, number of floors)

> **Example: Why Linear Regression Matters**  
> In structural design, engineers routinely use linear models to predict beam deflection from applied load, span length, and cross-section properties. The model coefficients directly correspond to physical parameters.

### Simple Linear Regression Review

**The Simplest Case: One Feature**

$$y = ax + b$$

where $a$ is the slope and $b$ is the intercept.

**Ordinary Least Squares (OLS):**

The OLS approach finds the line that minimizes the sum of squared residuals:

$$\text{minimize: } \sum_{i=1}^{n} (y_i - (ax_i + b))^2$$

**Key Properties:**
- Has a **unique closed-form solution** -- no iterative optimization needed
- The solution minimizes the total squared distance from data points to the line
- Computationally efficient: solved via matrix operations in $O(np^2)$ time
- Residuals (errors) sum to zero when an intercept is included

> **Example: Civil Engineering Interpretation**  
> If we model beam deflection as $\delta = a \cdot P + b$, then $a$ tells us how many mm the beam deflects per additional kN of load, and $b$ is the initial deflection under self-weight.

In [ ]:
# Simple Linear Regression -- quick review
from sklearn.linear_model import LinearRegression

# Generate sample data: y = 2*x - 5 + noise
rng = np.random.RandomState(1)
x = 10 * rng.rand(50)
y = 2 * x - 5 + rng.randn(50)

# Create and train the model
model = LinearRegression(fit_intercept=True)
model.fit(x[:, np.newaxis], y)

# Predictions for plotting
xfit = np.linspace(0, 10, 1000)
yfit = model.predict(xfit[:, np.newaxis])

print(f"Slope (a):     {model.coef_[0]:.4f}")    # ~2.0
print(f"Intercept (b): {model.intercept_:.4f}")   # ~-5.0

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(x, y, c='indianred', s=50, edgecolors='k', alpha=0.7, label='Data')
plt.plot(xfit, yfit, 'steelblue', linewidth=2, label='Linear Fit')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Simple Linear Regression', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Multiple Linear Regression

**Extending to Multiple Features:**

$$y = a_0 + a_1 x_1 + a_2 x_2 + \cdots + a_n x_n$$

**Geometric Interpretation:**
- With 1 feature: fitting a **line** in 2D
- With 2 features: fitting a **plane** in 3D
- With $n$ features: fitting a **hyperplane** in $(n+1)$-dimensional space

> **Example: CE Application**  
> Predicting beam deflection from multiple factors:
>
> $$\delta = a_0 + a_1 \cdot \text{load} + a_2 \cdot \text{span} + a_3 \cdot \text{moment\_of\_inertia}$$
>
> Each coefficient $a_i$ quantifies the independent effect of that variable on deflection.

**Important Note:**

"Linear" in Linear Regression refers to **linearity in the coefficients** $a_i$, not in the features $x_i$. This distinction becomes crucial when we discuss basis function regression below.

In [ ]:
# Multiple Linear Regression with 3 features
rng = np.random.RandomState(1)
X_multi = 10 * rng.rand(100, 3)  # 100 samples, 3 features
y_multi = 0.5 + np.dot(X_multi, [1.5, -2., 1.])  # true coefficients

model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

print(f"Intercept: {model_multi.intercept_:.4f}")   # ~0.5
print(f"Coefficients: {model_multi.coef_}")           # ~[1.5, -2., 1.]
print("\nLinear regression perfectly recovers the true coefficients!")

### Basis Function Regression

**Adapting Linear Regression to Nonlinear Relationships:**

The key idea is to **transform the input features** using basis functions $f_n(x)$, then fit a linear model in the transformed feature space.

**General Form:**

$$y = a_0 + a_1 f_1(x) + a_2 f_2(x) + a_3 f_3(x) + \cdots$$

**Why this works:**
- The model is still **linear in the coefficients** $a_i$
- But can capture **nonlinear relationships** between $x$ and $y$
- All the fast linear algebra machinery of OLS still applies
- The choice of basis functions encodes our assumptions about the data

**Common Basis Functions:**
- **Polynomial:** $f_n(x) = x^n$ (global influence)
- **Gaussian:** $f_n(x) = \exp(-(x - \mu_n)^2 / 2\sigma^2)$ (local influence)
- **Fourier:** $f_n(x) = \sin(n \omega x)$ or $\cos(n \omega x)$ (periodic patterns)

> **Key Insight: Power of Basis Functions**  
> By choosing appropriate basis functions, we can fit complex nonlinear patterns while still using fast linear regression algorithms.

### Polynomial Basis Functions

**If $f_n(x) = x^n$, the model becomes a polynomial:**

$$y = a_0 + a_1 x + a_2 x^2 + a_3 x^3 + \cdots + a_d x^d$$

**Degree $d$ controls model flexibility:**
- $d = 1$: straight line (underfitting if data is curved)
- $d = 2$: parabola
- $d = 7$: highly flexible curve
- $d = n-1$: passes through every data point (overfitting)

**Feature Transformation Example (degree 3):**

| **Input** | **Transformed Features** |
|---|---|
| $x = 2$ | $[1, 2, 4, 8]$ |
| $x = 3$ | $[1, 3, 9, 27]$ |

Built into scikit-learn as `PolynomialFeatures`. Note that a degree $d$ polynomial on $n$ features creates $\binom{n+d}{d}$ new features, which can grow rapidly.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Generate nonlinear data (sine wave with noise)
rng = np.random.RandomState(1)
x = 10 * rng.rand(50)
y = np.sin(x) + 0.1 * rng.randn(50)

xfit = np.linspace(0, 10, 1000)

# Create a 7th-degree polynomial model using pipeline
poly_model = make_pipeline(
    PolynomialFeatures(degree=7),
    LinearRegression()
)

poly_model.fit(x[:, np.newaxis], y)
yfit = poly_model.predict(xfit[:, np.newaxis])

plt.figure(figsize=(10, 6))
plt.scatter(x, y, c='indianred', s=50, edgecolors='k', alpha=0.7, label='Data')
plt.plot(xfit, yfit, 'steelblue', linewidth=2, label='Degree 7 Polynomial')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Polynomial Basis Function Regression', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Pipeline advantage: Combines preprocessing and modeling in one object")

### Gaussian Basis Functions

**An Alternative to Polynomials:**

Each basis function is a bell-shaped Gaussian centered at $\mu_i$:

$$f_i(x) = \exp\left(-\frac{(x - \mu_i)^2}{2\sigma^2}\right)$$

**Model Form:**

$$y = \sum_{i} a_i \cdot \exp\left(-\frac{(x - \mu_i)^2}{2\sigma^2}\right)$$

**Properties:**
- **Local influence:** each basis only affects predictions near its center $\mu_i$
- **Width controlled by $\sigma$:** larger $\sigma$ means broader, smoother bases
- **Smoother than high-degree polynomials:** no wild oscillations at boundaries
- **Uniform spacing:** centers $\mu_i$ are typically spread evenly across the data range

> **Key Insight: Local vs. Global**  
> Gaussian bases provide local, smooth approximations without the wild oscillations of high-degree polynomials. This makes them particularly useful for modeling structural response curves that vary smoothly across their domain.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class GaussianFeatures(BaseEstimator, TransformerMixin):
    """Uniformly spaced Gaussian features for 1D input."""
    
    def __init__(self, N, width_factor=2.0):
        self.N = N
        self.width_factor = width_factor
    
    def fit(self, X, y=None):
        # Create N centers along data range
        self.centers_ = np.linspace(X.min(), X.max(), self.N)
        self.width_ = self.width_factor * (self.centers_[1] - self.centers_[0])
        return self
    
    def transform(self, X):
        # Apply Gaussian basis functions
        return np.exp(-0.5 * ((X - self.centers_) / self.width_) ** 2)

# Demonstrate with 20 Gaussian bases
gauss_model = make_pipeline(GaussianFeatures(20), LinearRegression())
gauss_model.fit(x[:, np.newaxis], y)
yfit_gauss = gauss_model.predict(xfit[:, np.newaxis])

plt.figure(figsize=(10, 6))
plt.scatter(x, y, c='indianred', s=50, edgecolors='k', alpha=0.7, label='Data')
plt.plot(xfit, yfit_gauss, 'steelblue', linewidth=2, label='20 Gaussian Bases')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Gaussian Basis Function Regression', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### The Problem of Overfitting

**When the Model is Too Flexible:**

With too many basis functions, the model becomes overly flexible and starts fitting the **noise** in the data rather than the underlying pattern.

**Symptoms of Overfitting:**
- Excellent training performance but poor test performance
- Wild oscillations between data points
- Very large coefficient values (positive and negative)
- Model predictions are unstable -- small changes in data cause large changes in predictions

> **Example: CE Context**  
> Imagine fitting a 20th-degree polynomial to 25 soil bearing capacity measurements. The curve would pass near every point but oscillate wildly between them, giving nonsensical predictions at new locations.

**The Solution: Regularization**

Add a **penalty term** to the loss function that discourages large coefficients. This constrains the model's flexibility and improves generalization to unseen data.

In [ ]:
# Demonstrate overfitting with too many Gaussian bases
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

n_bases_list = [10, 20, 30]
titles = ['10 Bases (Good)', '20 Bases (Good)', '30 Bases (Overfitting)']

for ax, n_bases, title in zip(axes, n_bases_list, titles):
    gauss = make_pipeline(GaussianFeatures(n_bases), LinearRegression())
    gauss.fit(x[:, np.newaxis], y)
    yfit_g = gauss.predict(xfit[:, np.newaxis])
    
    ax.scatter(x, y, c='indianred', s=40, edgecolors='k', alpha=0.7)
    ax.plot(xfit, yfit_g, 'steelblue', linewidth=2)
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Ridge Regression (L2 Regularization)

**Penalizing Large Coefficients:**

Ridge regression adds a penalty proportional to the **sum of squared coefficients**:

$$\text{Loss} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \theta_j^2$$

**The Regularization Parameter $\alpha$:**
- $\alpha = 0$: standard linear regression (no regularization)
- Small $\alpha$: mild regularization, coefficients slightly shrunk
- Large $\alpha$: strong regularization, coefficients shrink toward zero
- $\alpha \to \infty$: all coefficients approach zero (underfitting)

**Key Properties:**
- Can be computed as efficiently as standard linear regression (closed-form solution)
- All coefficients are shrunk but **none become exactly zero**
- Also known as **Tikhonov regularization** in applied mathematics
- Equivalent to adding a Gaussian prior on the coefficients (Bayesian interpretation)

In [ ]:
from sklearn.linear_model import Ridge

# Ridge regression tames overfitting
ridge_model = make_pipeline(GaussianFeatures(30), Ridge(alpha=0.1))
ridge_model.fit(x[:, np.newaxis], y)
yfit_ridge = ridge_model.predict(xfit[:, np.newaxis])

plt.figure(figsize=(10, 6))
plt.scatter(x, y, c='indianred', s=50, edgecolors='k', alpha=0.7, label='Data')
plt.plot(xfit, yfit_ridge, 'steelblue', linewidth=2, label='Ridge (α=0.1, 30 bases)')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Ridge Regression: Taming Overfitting', fontsize=14)
plt.legend(fontsize=11)
plt.ylim(-1.5, 1.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Ridge regression produces a smooth fit even with 30 basis functions!")

### Lasso Regression (L1 Regularization)

**An Alternative Penalty Using Absolute Values:**

$$\text{Loss} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |\theta_j|$$

**Key Difference from Ridge:**

Lasso drives some coefficients to **exactly zero**, effectively performing **automatic feature selection**. The resulting model is **sparse** -- it uses only a subset of the available features.

**Why Does L1 Produce Zeros?**
- The absolute value function has a "corner" at zero
- During optimization, coefficients are pushed toward zero and can reach it exactly
- Ridge (L2) only shrinks coefficients -- they approach zero but never reach it

> **Example: CE Application**  
> When predicting bridge condition from 50 sensor measurements, Lasso automatically identifies the 5-10 most informative sensors, simplifying both the model and the monitoring system.

In [ ]:
from sklearn.linear_model import Lasso

# Lasso regression: automatic feature selection
lasso_model = make_pipeline(GaussianFeatures(30), Lasso(alpha=0.001))
lasso_model.fit(x[:, np.newaxis], y)
yfit_lasso = lasso_model.predict(xfit[:, np.newaxis])

# Check sparsity
coefficients = lasso_model.named_steps['lasso'].coef_
n_nonzero = np.sum(coefficients != 0)

plt.figure(figsize=(10, 6))
plt.scatter(x, y, c='indianred', s=50, edgecolors='k', alpha=0.7, label='Data')
plt.plot(xfit, yfit_lasso, 'steelblue', linewidth=2, 
         label=f'Lasso (α=0.001, {n_nonzero}/{len(coefficients)} bases used)')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Lasso Regression: Automatic Feature Selection', fontsize=14)
plt.legend(fontsize=11)
plt.ylim(-1.5, 1.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Non-zero coefficients: {n_nonzero} / {len(coefficients)}")
print("Lasso automatically selected the most relevant basis functions!")

### Ridge vs. Lasso: Choosing the Right Regularization

| | **Ridge (L2)** | **Lasso (L1)** |
|---|---|---|
| **Penalty** | $\alpha \sum \theta_j^2$ | $\alpha \sum |\theta_j|$ |
| **Coefficients** | All non-zero (shrunk) | Many exactly zero (sparse) |
| **Feature selection** | No -- keeps all features | Yes -- automatic selection |
| **Best when** | Many features all contribute | Few features truly matter |
| **Correlated features** | Shares weight among them | Picks one, zeros out others |
| **Solution** | Closed-form | Iterative (coordinate descent) |
| **Scikit-learn** | `Ridge(alpha=...)` | `Lasso(alpha=...)` |

**Elastic Net: The Best of Both Worlds**

Combines L1 and L2 penalties:

$$\text{Loss} = \sum (y_i - \hat{y}_i)^2 + \alpha_1 \sum |\theta_j| + \alpha_2 \sum \theta_j^2$$

Available as `ElasticNet` in scikit-learn. Useful when you want some sparsity but also want to handle correlated features gracefully.

> **Key Insight: Practical Guidance**  
> Start with Ridge if you have no reason to expect sparsity. Use Lasso if you suspect only a few features matter. Use Elastic Net when features are correlated and you want some sparsity.

---
## 6. Summary and Next Steps

### Summary: Naive Bayes Classification

**What We Learned:**

1. **Naive Bayes Fundamentals**
   - Based on Bayes's theorem with naive independence assumption
   - Generative model that learns $P(\text{features} \mid \text{class})$
   - Fast, simple, and effective for many problems

2. **Gaussian Naive Bayes**
   - Assumes features follow Gaussian (normal) distributions
   - Suitable for continuous numerical features
   - Applied to structural health classification

3. **Multinomial Naive Bayes**
   - Designed for discrete count data
   - Excellent for text classification
   - Used with TF-IDF feature extraction

4. **Key Strengths**
   - Very fast training and prediction
   - Works well with high-dimensional data
   - Provides probabilistic predictions
   - Few hyperparameters to tune

5. **Limitations**
   - Assumes feature independence (often violated)
   - Can be outperformed by more complex models
   - Sensitive to feature distribution assumptions

### Next Steps

**Building on These Foundations:**

**Topics to Explore:**
- Logistic Regression (discriminative alternative to Naive Bayes)
- Support Vector Machines (more flexible decision boundaries)
- Decision Trees and Ensemble Methods (Random Forest, Gradient Boosting)
- Deep Learning for text (Word embeddings, Transformers)
- Advanced regression: Generalized Linear Models (GLMs)
- Time series modeling and forecasting

**Practice Exercises:**
- Implement Gaussian NB on civil engineering datasets
- Build polynomial regression for nonlinear structural behavior
- Compare regularization methods (Ridge, Lasso, Elastic Net)
- Feature engineering for real-world prediction problems
- Cross-validation and hyperparameter tuning

---

**References:**
- Python Data Science Handbook by Jake VanderPlas, Chapter 5.05
- Scikit-learn documentation: https://scikit-learn.org/stable/modules/naive_bayes.html
- 20 Newsgroups dataset: http://qwone.com/~jason/20Newsgroups/

---

**End of Week 07 Lecture Notebook**